In [0]:
# DBTITLE 1,Load CSV files with Auto Loader to bronze table
# Read CSV files using Auto Loader

df = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "csv")
  .option("cloudFiles.schemaLocation", "/Volumes/retail_postgres/volume/blob_source/_schema")
  .option("header", "true")
  .option("inferSchema", "true")
  .load("/Volumes/retail_postgres/volume/blob_source/transaction_source/")
)

###write to bronze table -process available data and stop

In [0]:
# Create the blob_bronze schema if it doesn't exist
spark.sql("CREATE SCHEMA IF NOT EXISTS retail_postgres.blob_bronze")

In [0]:
query = (df.writeStream
  .option("checkpointLocation", "/Volumes/retail_postgres/volume/blob_source/_checkpoint")
  .trigger(availableNow=True)
  .toTable("retail_postgres.blob_bronze.transactions")
)

In [0]:
%sql
select count(*) from retail_postgres.blob_bronze.transactions

In [0]:
# Wait for the batch to complete
query.awaitTermination()